# 01 · Synthetic images & HSV fire segmentation
* **Synthetic images** - you can skip generation if you downloaded the authors' Mendeley set in notebook 00. The optional cells below regenerate your own with Stable Diffusion (paper: 1,020 fire + 1,181 non-fire, prompts *"a fire in a forest"* / *"a forest without fire"*, CLIP score 35.6-45.2).
* **Segmentation** - Eq. 1-5 of the paper (HSV mask → Gaussian blur → morphological close → Canny → green overlay). This is only a visualisation, it is *not* used for training.

In [ ]:
# ---- Colab setup (run first; if you ever see Keras-3 errors: Runtime > Restart session, then re-run) ----
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # the paper's code style needs Keras 2
!pip -q install tf_keras
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/firenet"       # everything lives here, so a disconnect loses nothing
sys.path.insert(0, PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from types import SimpleNamespace
from IPython.display import Image, display
import firenet as fn                             # written to Drive by 00_setup_and_data.ipynb
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATASETS = ["A", "B"]          # which real dataset(s)
MODES    = ["paper", "strict"] # paper = split AFTER adding synthetic (as in the paper); strict = synthetic only in training folds
EPOCHS   = 2                   # <-- smoke test first! then set to 50 for the real run

cfg = SimpleNamespace(data=f"{PROJECT}/data", dataset="B", mode="paper", no_synth=False,
                      epochs=EPOCHS, bs=32, lr=1e-4, seed=42, out=f"{PROJECT}/runs")
def get(d):
    return fn.load_cached(cfg.data, d, use_synth=True, cache_dir=f"{PROJECT}/cache")

## Look at the data

In [ ]:
def grid(X, y, title, n=5, seed=0):
    rng = np.random.default_rng(seed); fig, ax = plt.subplots(2, n, figsize=(2.6 * n, 5.4))
    for r, lab in enumerate((1, 0)):
        for c, i in enumerate(rng.choice(np.where(y == lab)[0], n, replace=False)):
            ax[r, c].imshow(X[i]); ax[r, c].set_title("Fire" if lab else "No Fire", fontsize=9); ax[r, c].axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

for d in ("A", "B"):
    Xr, yr, sr = fn.load_cached(cfg.data, d, use_synth=False, cache_dir=f"{PROJECT}/cache")
    grid(Xr, yr, f"Dataset {d} (real)")
Xa, ya, sa = fn.load_cached(cfg.data, "A", use_synth=True, cache_dir=f"{PROJECT}/cache")
Xs, ys = Xa[sa], ya[sa]
grid(Xs, ys, "Synthetic (Stable Diffusion)")

## Fire segmentation (Fig. 6 / 7 of the paper)

In [ ]:
for d in ("A", "B"):
    Xr, yr, _ = fn.load_cached(cfg.data, d, use_synth=False, cache_dir=f"{PROJECT}/cache")
    idx = np.where(yr == 1)[0][:5]
    fig, ax = plt.subplots(2, 5, figsize=(14, 6))
    for j, i in enumerate(idx):
        ax[0, j].imshow(Xr[i]); ax[0, j].set_title("Original Image", fontsize=8)
        ax[1, j].imshow(fn.segment_fire(Xr[i])); ax[1, j].set_title("Detected Fire Edges", fontsize=8)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    fig.suptitle(f"Fire segmentation - Dataset {d}"); plt.tight_layout(); plt.show()

## (Optional) generate your own synthetic images
Needs ~15 GB of GPU-time-friendly downloads (SD 1.5 weights). ~2,200 images take roughly 30-60 min on a T4 at 50 steps. **Skip this if you already have the Mendeley synthetic set.**
Write them to a *different* folder (`synthetic_own`) so you don't mix them with the authors' images.

In [ ]:
GENERATE = False
if GENERATE:
    !pip -q install diffusers transformers accelerate torchmetrics
    !python {PROJECT}/generate_synthetic.py --out {PROJECT}/data/synthetic_own --n_fire 1020 --n_nofire 1181 --clip_score --diverse